# 04 - Pretraining

Three pretraining experiments (requires GPU — use Colab A100):

- **Model A**: Direct transfer — load insect BarcodeMamba, skip to fine-tune
- **Model B**: Train from scratch on marine fish barcodes
- **Model C**: Domain-adaptive — continue pretraining insect model on fish (RECOMMENDED)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kluless/marinemamba/blob/main/notebooks/04_pretrain.ipynb)

In [ ]:
# === COLAB SETUP (A100 runtime required) ===
!pip install torch==2.3.1 --index-url https://download.pytorch.org/whl/cu121
!pip install lightning==2.3.3 hydra-core==1.3.2 omegaconf==2.3.0
!pip install transformers==4.42.3 einops==0.8.0 timm==1.0.7
!pip install torchtext==0.18.0 scikit-learn==1.3.2 wandb pandas tqdm

# Install Mamba SSM (CUDA 12.1)
!pip install mamba-ssm==2.2.1 causal-conv1d==1.3.0.post1

# Clone repos
!git clone https://github.com/kluless/marinemamba.git
!git clone https://github.com/bioscan-ml/BarcodeMamba.git

%cd marinemamba

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
import sys
sys.path.insert(0, "../BarcodeMamba")

from pathlib import Path
import json

PROC_DIR = Path("../data/processed")
CKPT_DIR = Path("../checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Load dataset stats
with open(PROC_DIR / "dataset_stats.json") as f:
    stats = json.load(f)
print(f"Dataset: {stats['pretrain_size']} pretrain, {stats['n_classes']} species")

## Download BarcodeMamba Pretrained Weights

In [ ]:
from huggingface_hub import hf_hub_download

# Download the mini k-mer model (7.4M params, best efficiency)
model_name = "BarcodeMamba-dim384-layer2-k-mer"
ckpt_path = hf_hub_download(
    repo_id="bioscan-ml/BarcodeMamba",
    filename=f"{model_name}/last.ckpt",
    local_dir=CKPT_DIR / "barcodemamba_insect"
)
print(f"Downloaded insect checkpoint: {ckpt_path}")

## Model A: Direct Transfer (just save the insect checkpoint path)

In [ ]:
# Model A is just the insect checkpoint — no additional pretraining
model_a_path = ckpt_path
print(f"Model A (direct transfer): {model_a_path}")

## Model B: Pretrain from Scratch on Fish

Initialize a fresh model and pretrain on marine fish barcodes with NTP.

In [ ]:
# TODO: Adapt BarcodeMamba training script for marine data
# This cell runs the pretraining from scratch
#
# Key config changes from BarcodeMamba defaults:
#   dataset.input_path = ../data/processed/
#   dataset.pretrain_file = pre_training.csv
#   train.pretrained_model_path = None  (fresh init)
#   train.max_epochs = 50
#   train.lr = 8e-4
#
# Command (from BarcodeMamba repo):
# python train.py \
#   dataset=marine_fish_pretrain \
#   dataset.input_path=../marinemamba/data/processed/ \
#   tokenizer=k_mer \
#   train.max_epochs=50

print("TODO: Run scratch pretraining — see BarcodeMamba train.py")
print("Estimated time: 4-12 hours on A100")
# model_b_path = CKPT_DIR / "model_b_scratch" / "last.ckpt"

## Model C: Domain-Adaptive Pretraining (RECOMMENDED)

Load insect checkpoint, continue pretraining on fish barcodes with lower LR.

In [ ]:
# TODO: Adapt BarcodeMamba training script for domain adaptation
# This cell runs continued pretraining from insect checkpoint
#
# Key config changes:
#   dataset.input_path = ../data/processed/
#   dataset.pretrain_file = pre_training.csv
#   train.pretrained_model_path = <insect_checkpoint>
#   train.max_epochs = 20        (fewer epochs — just adapting)
#   train.lr = 2e-4              (lower LR — don't destroy insect knowledge)
#
# Command:
# python train.py \
#   dataset=marine_fish_pretrain \
#   dataset.input_path=../marinemamba/data/processed/ \
#   tokenizer=k_mer \
#   train.pretrained_model_path=<insect_ckpt> \
#   train.max_epochs=20 \
#   train.lr=2e-4

print("TODO: Run domain-adaptive pretraining — see BarcodeMamba train.py")
print(f"Starting from: {model_a_path}")
print("Estimated time: 2-8 hours on A100")
# model_c_path = CKPT_DIR / "model_c_adapted" / "last.ckpt"

## Summary

After this notebook, you should have 3 checkpoints:

| Model | Checkpoint | Description |
|---|---|---|
| A | `barcodemamba_insect/last.ckpt` | Insect pretrained (direct transfer) |
| B | `model_b_scratch/last.ckpt` | Scratch pretrained on fish |
| C | `model_c_adapted/last.ckpt` | Insect → fish domain adapted |

## Next

Proceed to `05_finetune.ipynb` to fine-tune all models and compare.